In [ ]:
# -*- coding: utf-8 -*-
"""
LWE pipeline with xarray
Author: <you>
Requires: xarray, numpy, pandas (dask optional but supported via open_mfdataset)
"""







#from scipy.ndimage import binary_closing
#import dask
#from dask.diagnostics import ProgressBar

#import os, glob, re, math, gc, time
#from dataclasses import dataclass
# from tqdm import tqdm




README
======

Purpose
-------
This script detects and exports low-wind events (LWEs) from hourly wind-speed
data. The workflow is designed to identify persistent low-wind periods at each
grid cell and to save the resulting event catalogue and LWE flag fields.

Definition of Low-Wind Events
-----------------------------
In this script, a baseline low-wind event is first defined using method M0:

    M0: consecutive hours with U10 < 3 m/s lasting at least 48 hours,
        with no gap allowed inside the event.

Here, U10 denotes the 10 m wind speed.

After all baseline M0 events are detected, a gap-merging procedure is applied:

    Adjacent M0 events are merged into one longer event if the time gap
    between them is less than or equal to 6 hours.

Therefore, the final LWE events used in this script are persistent near-surface
low-wind events that satisfy U10 < 3 m/s for at least 48 hours initially, with
neighbouring long events merged when they are separated by a short interruption
of no more than 6 hours.

Main Outputs
------------
The script produces event-level information, including:

    - latitude and longitude of the ERA5 grid-cell centre
    - event start time
    - event end time
    - event duration in hours
    - event type / label
    - selected event-level statistics, such as median wind-speed difference,
      median boundary layer height, and median surface sensible heat flux

Time Convention
---------------
All timestamps are assumed to be in UTC, consistent with ERA5 hourly data.

User Configuration
------------------
Before running the script, users should modify the configuration section,
including:

    - output directory
    - input xarray DataArrays
    - processing years
    - wind-speed threshold
    - minimum event duration
    - allowed gap length for merging
    - spatial and temporal chunk sizes
"""

In [6]:
# -*- coding: utf-8 -*-

import os, gc, glob
from pathlib import Path
from __future__ import annotations
from typing import Tuple, Dict, List, Optional

import numpy as np
import pandas as pd
import xarray as xr


# Part 1: Detect LWE

In [2]:

def list_monthly_files(data_dir: str, base_glob: str) -> List[str]:
    '''Return sorted list of monthly NetCDF files in data_dir matching base_glob.'''
    files = sorted(glob.glob(os.path.join(data_dir, base_glob)))
    if not files:
        raise FileNotFoundError(f"No files matched: {os.path.join(data_dir, base_glob)}")

    def extract_year_month(path: str):
        m = re.search(r"(\d{4})_(\d{1,2})\.nc$", os.path.basename(path))
        if not m:
            raise ValueError(f"Filename does not match expected pattern: {path}")
        year, month = map(int, m.groups())
        return (year, month)
    
    files = sorted(files, key=extract_year_month)
    return files

def open_era5(files_glob: str, chunks: dict | None = None) -> xr.Dataset:
    ds = xr.open_mfdataset(
        files_glob,
        combine="by_coords",
        engine="h5netcdf", 
        parallel=False,
        chunks=chunks
    )
    # 统一时间维名
    if "valid_time" in ds.dims or "valid_time" in ds.coords:
        ds = ds.rename({"valid_time": "time"})
    
    return ds

def subset_region_time(ds: xr.Dataset, region: dict, time_win: slice) -> xr.Dataset:
    return ds.sel(
        time=time_win,
        latitude=slice(region["lat_max"], region["lat_min"]),  # ERA5 纬度降序
        longitude=slice(region["lon_min"], region["lon_max"]),
    )

def windspeed(ds: xr.Dataset, height: int) -> xr.DataArray:

    u_name = f"u{height}"
    v_name = f"v{height}"
    U = ((ds[u_name]**2 + ds[v_name]**2) ** 0.5).rename(f"U{height}").astype("float32")

    return U.transpose("time", "latitude", "longitude")

def drop_variable(U):
    U  = U.drop_vars("number", errors="ignore")
    U  = U.drop_vars("expver", errors="ignore")
    
    return U

def open_var(files_glob: str, variable_name: str, chunks: dict | None = None) -> xr.DataArray:
    # variable_name: blh, sshf
    ds = xr.open_mfdataset(
        files_glob,
        combine="by_coords",
        engine="h5netcdf",
        parallel=False,
        chunks="auto"
    )
    if "valid_time" in ds.dims or "valid_time" in ds.coords:
        ds = ds.rename({"valid_time": "time"})
    ds = ds[variable_name].astype("float32")
    ds = ds.drop_vars("number", errors="ignore")
    ds = ds.drop_vars("expver", errors="ignore")
    
    return ds

In [3]:
REGION = dict(lat_min=30.0, lat_max=75.0, lon_min=-25.0, lon_max=40.0)
TIME_WIN = slice("1979-01-01", "2024-12-31")


ds_10m = open_era5(r"D:\ERA5-uv-10m_30N_90N_80W_40E\*.nc".replace('\\', '/'), chunks={"time": 240})
ds_10m  = subset_region_time(ds_10m, REGION, TIME_WIN)
U10 = windspeed(ds_10m, 10)
U10 = drop_variable(U10)

ds_100m = open_era5(r"D:\ERA5-uv-100m_30N_90N_80W_40E\*.nc", chunks={"time": 240})
ds_100m  = subset_region_time(ds_100m, REGION, TIME_WIN)
U100 = windspeed(ds_100m, 100)
U100 = drop_variable(U100)

U100

<xarray.DataArray 'U100' (time: 403248, latitude: 181, longitude: 261)> Size: 76GB
dask.array<pow, shape=(403248, 181, 261), dtype=float32, chunksize=(248, 81, 159), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 3MB 1979-01-01 ... 2024-12-31T23:00:00
  * latitude   (latitude) float64 1kB 75.0 74.75 74.5 74.25 ... 30.5 30.25 30.0
  * longitude  (longitude) float64 2kB -25.0 -24.75 -24.5 ... 39.5 39.75 40.0

In [4]:
blh = open_var(r"D:\Data\ERA5_BLH_30N_75N_25W_40E\blh*.nc".replace("\\","/"), "blh", chunks={"time": 240})
sshf = open_var(r"D:\Data\ERA5_sshf_slhf_t2m_sp_30N_75N_25W_40E\extracted_sshf_slhf\sshf_slhf*.nc".replace("\\","/"), "sshf", chunks={"time": 240})

In [ ]:

# ============================================================
# USER CONFIGURATION
# ============================================================

# >>> USER INPUT REQUIRED <<<
# Output directory for the generated parquet and NetCDF files.
OUT_DIR = r"D:\test"

# >>> USER INPUT REQUIRED <<<
# Input xarray DataArrays must be loaded before running main().
# Required variables:
#   U10   : 10 m wind speed, with dimensions ("time", "latitude", "longitude")
#   U100  : 100 m wind speed, with dimensions ("time", "latitude", "longitude")
#   blh   : boundary layer height, with dimensions ("time", "latitude", "longitude")
#   sshf  : surface sensible heat flux, with dimensions ("time", "latitude", "longitude")
#
# Example:
# U10 = ds["U10"]
# U100 = ds["U100"]
# blh = ds["blh"]
# sshf = ds["sshf"]

# >>> USER INPUT REQUIRED <<<
# Chunking parameters. Adjust according to available memory.
TIME_WINDOW_DAYS = 365
LAT_STRIPE = 64
LON_STRIPE = 96

# >>> USER INPUT REQUIRED <<<
# LWE detection parameters.
THR = 3.0          # Wind speed threshold, unit: m s-1
MINLEN_H = 48      # Minimum event duration, unit: hours
TOL_H = 6          # Maximum gap allowed for merging adjacent long events, unit: hours
EPS = 1e-6         # Floating-point tolerance

# >>> USER INPUT REQUIRED <<<
# Time periods to process.
YEAR_SPANS = [
    (f"{year}-01-01", f"{year}-12-31")
    for year in range(1979, 1980)
]


# ============================================================
# BASIC FUNCTIONS
# ============================================================

def build_time_windows(time_index: pd.DatetimeIndex, window_days: int) -> list[tuple[int, int]]:
    """Split a time index into windows of a given number of days."""

    n = len(time_index)
    edges = [0]
    cur = time_index[0]

    while True:
        nxt = cur + pd.Timedelta(days=window_days)
        j = int(np.searchsorted(time_index.values, np.datetime64(nxt), side="left"))
        edges.append(min(j, n))

        if edges[-1] >= n:
            break

        cur = time_index[edges[-1]]

    return [(edges[i], edges[i + 1]) for i in range(len(edges) - 1)]


def keep_long_runs_2d(mask2d: np.ndarray, minlen_h: int) -> np.ndarray:
    """
    Keep only True segments with length >= minlen_h.

    Parameters
    ----------
    mask2d : ndarray
        Boolean array with shape (time, grid_point).
    minlen_h : int
        Minimum duration in hours.

    Returns
    -------
    ndarray
        Boolean array with the same shape as mask2d.
    """

    T, P = mask2d.shape
    out = np.zeros_like(mask2d, dtype=bool)

    for p in range(P):
        x = mask2d[:, p].astype(np.int8)

        if not x.any():
            continue

        diff = np.diff(np.r_[0, x, 0])
        starts = np.where(diff == 1)[0]
        ends = np.where(diff == -1)[0]

        lengths = ends - starts

        for s, e, L in zip(starts, ends, lengths):
            if L >= minlen_h:
                out[s:e, p] = True

    return out


def merge_only_if_both_long_2d(
    mask2d: np.ndarray,
    minlen_h: int,
    tol_h: int,
) -> np.ndarray:
    """
    Merge neighbouring long events only if the gap between them is <= tol_h.

    The input mask is assumed to contain only long events.
    Short events are not used for merging.
    """

    T, P = mask2d.shape
    out = mask2d.copy()

    for p in range(P):
        x = out[:, p].astype(np.int8)

        if not x.any():
            continue

        diff = np.diff(np.r_[0, x, 0])
        starts = np.where(diff == 1)[0]
        ends = np.where(diff == -1)[0]

        for i in range(len(starts) - 1):
            gap_len = starts[i + 1] - ends[i]

            if gap_len <= tol_h:
                out[ends[i]:starts[i + 1], p] = True

    return out


# ============================================================
# EVENT EXTRACTION AND EVENT-LEVEL STATISTICS
# ============================================================

def detect_with_stats(
    mask_blk_bool: np.ndarray,
    U10_blk_np: np.ndarray,
    U100_blk_np: np.ndarray,
    lat_vals: np.ndarray,
    lon_vals: np.ndarray,
    i0: int,
    j0: int,
    t_start: int,
    time_idx: pd.DatetimeIndex,
    label: str,
    minlen_h: int,
    BLH_blk_np: np.ndarray | None = None,
    SSHF_blk_np: np.ndarray | None = None,
) -> pd.DataFrame:
    """
    Extract LWE events and keep only selected event-level variables:

    - lat
    - lon
    - start
    - end
    - duration_h
    - kind
    - delta_u_median
    - blh_p50
    - sshf_p50
    """

    T, L, M = mask_blk_bool.shape

    base_cols = [
        "lat",
        "lon",
        "start",
        "end",
        "duration_h",
        "kind",
        "delta_u_median",
        "blh_p50",
        "sshf_p50",
    ]

    if T == 0 or L == 0 or M == 0:
        return pd.DataFrame(columns=base_cols)

    N = L * M
    mask2 = mask_blk_bool.reshape(T, N).astype(np.int8)

    # Detect the start and end of True segments.
    diff = np.diff(mask2, axis=0, prepend=0, append=0)

    s_t, s_n = np.where(diff == 1)
    e_t, e_n = np.where(diff == -1)

    # Pair starts and ends by grid column.
    order_s = np.lexsort((s_t, s_n))
    order_e = np.lexsort((e_t, e_n))

    s_t, s_n = s_t[order_s], s_n[order_s]
    e_t, e_n = e_t[order_e], e_n[order_e]

    mask_col = s_n == e_n
    s_t, e_t, ncol = s_t[mask_col], e_t[mask_col], s_n[mask_col]

    dur = e_t - s_t
    ok = dur >= minlen_h

    if not np.any(ok):
        return pd.DataFrame(columns=base_cols)

    # e_t is the first False index. Therefore, e_t - 1 is the last event hour.
    s_t, e_t, ncol, dur = s_t[ok], e_t[ok] - 1, ncol[ok], dur[ok]

    jj = ncol % M
    ii = ncol // M

    lats = lat_vals[i0 + ii]
    lons = lon_vals[j0 + jj]

    starts_abs = time_idx[t_start + s_t]
    ends_abs = time_idx[t_start + e_t]

    n_evt = len(s_t)

    delta_u_median = np.empty(n_evt, dtype=np.float32)
    blh_p50 = np.full(n_evt, np.nan, dtype=np.float32)
    sshf_p50 = np.full(n_evt, np.nan, dtype=np.float32)

    for k in range(n_evt):
        t0, t1 = s_t[k], e_t[k]
        ii_k, jj_k = ii[k], jj[k]

        seg10 = U10_blk_np[t0:t1 + 1, ii_k, jj_k]
        seg100 = U100_blk_np[t0:t1 + 1, ii_k, jj_k]

        seg_delta_u = seg100 - seg10
        delta_u_median[k] = float(np.nanmedian(seg_delta_u))

        if BLH_blk_np is not None:
            seg_blh = BLH_blk_np[t0:t1 + 1, ii_k, jj_k]
            blh_p50[k] = float(np.nanpercentile(seg_blh, 50))

        if SSHF_blk_np is not None:
            seg_sshf = SSHF_blk_np[t0:t1 + 1, ii_k, jj_k]

            # Convert accumulated ERA5 SSHF from J m-2 to W m-2.
            # The sign convention follows the original workflow.
            seg_sshf = seg_sshf / (-3600)

            sshf_p50[k] = float(np.nanpercentile(seg_sshf, 50))

    out = pd.DataFrame({
        "lat": lats.astype(float),
        "lon": lons.astype(float),
        "start": starts_abs,
        "end": ends_abs,
        "duration_h": dur.astype(np.int32),
        "kind": label,
        "delta_u_median": delta_u_median,
        "blh_p50": blh_p50,
        "sshf_p50": sshf_p50,
    })

    return out


# ============================================================
# LWE FLAG CLASSIFICATION
# ============================================================

def make_LWE_flag(
    low10_3d: np.ndarray,
    low100_3d: np.ndarray,
    high10_3d: np.ndarray,
    minlen_h: int,
    tol_h: int,
):
    """
    Generate integer LWE flags based on U10 and U100.

    Flag definitions:

    0 - No low-wind condition:
        All hours not covered by flags 1-5.

    1 - Hub-height-only low wind:
        U10 >= threshold and U100 < threshold.

    2 - Shallow-LWE-like event, short:
        U10 < threshold and U100 >= threshold, duration < minlen_h.

    3 - Deep-LWE-like event, short:
        U10 < threshold and U100 < threshold, duration < minlen_h.

    4 - Shallow LWE:
        U10 < threshold and U100 >= threshold, duration >= minlen_h.

    5 - Deep LWE:
        U10 < threshold and U100 < threshold, duration >= minlen_h.

    Returns
    -------
    flag_3d : ndarray
        Integer flag array with shape (time, latitude, longitude).
    deep_merged : ndarray
        Boolean mask of final deep LWEs.
    shallow_merged : ndarray
        Boolean mask of final shallow LWEs.
    """

    T, L, M = low10_3d.shape

    shallow_raw = low10_3d & (~low100_3d)
    deep_raw = low10_3d & low100_3d
    hub_raw = high10_3d & low100_3d

    def _split_short_long(raw3d: np.ndarray, minlen_h: int, tol_h: int):
        """
        Split raw low-wind periods into short events and persistent merged events.
        """

        T, L, M = raw3d.shape
        raw2d = raw3d.reshape(T, -1)

        long2d = keep_long_runs_2d(raw2d, minlen_h)
        short2d = raw2d & (~long2d)
        merged2d = merge_only_if_both_long_2d(long2d, minlen_h, tol_h)

        short3d = short2d.reshape(T, L, M)
        merged3d = merged2d.reshape(T, L, M)

        return short3d, merged3d

    shallow_short, shallow_merged = _split_short_long(
        shallow_raw,
        minlen_h,
        tol_h,
    )

    deep_short, deep_merged = _split_short_long(
        deep_raw,
        minlen_h,
        tol_h,
    )

    flag_3d = np.zeros((T, L, M), dtype=np.int8)

    flag_3d[hub_raw] = 1
    flag_3d[shallow_short] = 2
    flag_3d[deep_short] = 3
    flag_3d[shallow_merged] = 4
    flag_3d[deep_merged] = 5

    return flag_3d, deep_merged, shallow_merged


# ============================================================
# MAIN WORKFLOW
# ============================================================

def main():

    os.makedirs(OUT_DIR, exist_ok=True)

    for span_start, span_end in YEAR_SPANS:

        span_tag = f"{span_start[:4]}_{span_end[:4]}"
        OUT_DIR_SPAN = os.path.join(OUT_DIR, span_tag)
        os.makedirs(OUT_DIR_SPAN, exist_ok=True)

        done_flag = os.path.join(OUT_DIR_SPAN, f".done_{span_tag}")

        if os.path.exists(done_flag):
            print(f"[{span_tag}] already done. Skip.")
            continue

        print(f"[{span_tag}] processing...")

        TIME_WIN = slice(span_start, span_end)

        # Select the current time span.
        U10_sel = U10.sel(time=TIME_WIN)
        U100_sel = U100.sel(time=TIME_WIN)
        BLH_sel = blh.sel(time=TIME_WIN)
        SSHF_sel = sshf.sel(time=TIME_WIN)

        if U10_sel.time.size == 0:
            print(f"[{span_tag}] no data in this span. Skip.")
            Path(done_flag).write_text("ok", encoding="utf-8")
            continue

        time_idx = pd.DatetimeIndex(U10_sel.time.values)
        lat_vals = U10_sel.latitude.values
        lon_vals = U10_sel.longitude.values

        assert U10_sel.sizes == U100_sel.sizes

        ntime = U10_sel.sizes["time"]
        nlat = U10_sel.sizes["latitude"]
        nlon = U10_sel.sizes["longitude"]

        windows = build_time_windows(time_idx, TIME_WINDOW_DAYS)

        # Extra time buffer used to avoid cutting events at window boundaries.
        OVERLAP = MINLEN_H + TOL_H

        for wi, (t_start, t_stop) in enumerate(windows, start=1):

            core_start, core_stop = t_start, t_stop
            ext_start = max(0, core_start - OVERLAP)
            ext_stop = min(len(time_idx), core_stop + OVERLAP)
            time_slice_ext = slice(ext_start, ext_stop)

            for i0 in range(0, nlat, LAT_STRIPE):

                i1 = min(i0 + LAT_STRIPE, nlat)

                for j0 in range(0, nlon, LON_STRIPE):

                    j1 = min(j0 + LON_STRIPE, nlon)

                    fn_deep = os.path.join(
                        OUT_DIR_SPAN,
                        f"deep_t{t_start:07d}_{t_stop:07d}_"
                        f"lat{i0:04d}_{i1:04d}_lon{j0:04d}_{j1:04d}.parquet",
                    )

                    fn_shal = os.path.join(
                        OUT_DIR_SPAN,
                        f"shallow_t{t_start:07d}_{t_stop:07d}_"
                        f"lat{i0:04d}_{i1:04d}_lon{j0:04d}_{j1:04d}.parquet",
                    )

                    fn_hubl = os.path.join(
                        OUT_DIR_SPAN,
                        f"hubonlylow_t{t_start:07d}_{t_stop:07d}_"
                        f"lat{i0:04d}_{i1:04d}_lon{j0:04d}_{j1:04d}.parquet",
                    )

                    fn_flag = os.path.join(
                        OUT_DIR_SPAN,
                        f"LWEflag_t{core_start:07d}_{core_stop:07d}_"
                        f"lat{i0:04d}_{i1:04d}_lon{j0:04d}_{j1:04d}.nc",
                    )

                    # Skip the block if all output files already exist.
                    if (
                        os.path.exists(fn_deep)
                        and os.path.exists(fn_shal)
                        and os.path.exists(fn_hubl)
                        and os.path.exists(fn_flag)
                    ):
                        continue

                    # Load one data block into memory.
                    U10_blk = (
                        U10_sel
                        .isel(
                            time=time_slice_ext,
                            latitude=slice(i0, i1),
                            longitude=slice(j0, j1),
                        )
                        .transpose("time", "latitude", "longitude")
                        .load()
                    )

                    U100_blk = (
                        U100_sel
                        .isel(
                            time=time_slice_ext,
                            latitude=slice(i0, i1),
                            longitude=slice(j0, j1),
                        )
                        .transpose("time", "latitude", "longitude")
                        .load()
                    )

                    BLH_blk = (
                        BLH_sel
                        .isel(
                            time=time_slice_ext,
                            latitude=slice(i0, i1),
                            longitude=slice(j0, j1),
                        )
                        .transpose("time", "latitude", "longitude")
                        .load()
                    )

                    SSHF_blk = (
                        SSHF_sel
                        .isel(
                            time=time_slice_ext,
                            latitude=slice(i0, i1),
                            longitude=slice(j0, j1),
                        )
                        .transpose("time", "latitude", "longitude")
                        .load()
                    )

                    U10_np = U10_blk.values
                    U100_np = U100_blk.values
                    BLH_np = BLH_blk.values
                    SSHF_np = SSHF_blk.values

                    assert U10_np.shape == U100_np.shape == BLH_np.shape == SSHF_np.shape

                    # Build basic threshold masks.
                    low10 = np.isfinite(U10_np) & (U10_np < THR + EPS)
                    low100 = np.isfinite(U100_np) & (U100_np < THR + EPS)
                    high10 = np.isfinite(U10_np) & (U10_np >= THR + EPS)

                    # Generate LWE flags and final shallow/deep masks.
                    LWE_flag_ext, deep_merged, shallow_merged = make_LWE_flag(
                        low10,
                        low100,
                        high10,
                        MINLEN_H,
                        TOL_H,
                    )

                    # Build hub-height-only low-wind events.
                    hubonlylow_raw = high10 & low100

                    raw2d = hubonlylow_raw.reshape(hubonlylow_raw.shape[0], -1)
                    long2d = keep_long_runs_2d(raw2d, MINLEN_H)
                    merged2d = merge_only_if_both_long_2d(long2d, MINLEN_H, TOL_H)
                    hubonlylow_merged = merged2d.reshape(hubonlylow_raw.shape)

                    # Extract event-level statistics.
                    df_deep = detect_with_stats(
                        deep_merged,
                        U10_np,
                        U100_np,
                        lat_vals,
                        lon_vals,
                        i0,
                        j0,
                        ext_start,
                        time_idx,
                        "deep",
                        MINLEN_H,
                        BLH_np,
                        SSHF_np,
                    )

                    df_shal = detect_with_stats(
                        shallow_merged,
                        U10_np,
                        U100_np,
                        lat_vals,
                        lon_vals,
                        i0,
                        j0,
                        ext_start,
                        time_idx,
                        "shallow",
                        MINLEN_H,
                        BLH_np,
                        SSHF_np,
                    )

                    df_hubl = detect_with_stats(
                        hubonlylow_merged,
                        U10_np,
                        U100_np,
                        lat_vals,
                        lon_vals,
                        i0,
                        j0,
                        ext_start,
                        time_idx,
                        "hubonlylow",
                        MINLEN_H,
                        BLH_np,
                        SSHF_np,
                    )

                    # Keep only events whose start time lies within the core window.
                    core_start_time = time_idx[core_start]

                    if core_stop < len(time_idx):
                        core_stop_time_right = time_idx[core_stop]
                    else:
                        step = (
                            time_idx[1] - time_idx[0]
                            if len(time_idx) > 1
                            else pd.Timedelta(hours=1)
                        )
                        core_stop_time_right = time_idx[-1] + step

                    def keep_core(df: pd.DataFrame) -> pd.DataFrame:
                        if df.empty:
                            return df

                        return df[
                            (df["start"] >= core_start_time)
                            & (df["start"] < core_stop_time_right)
                        ]

                    df_deep = keep_core(df_deep)
                    df_shal = keep_core(df_shal)
                    df_hubl = keep_core(df_hubl)

                    # Save LWE flag for the core time window.
                    t0_core = core_start - ext_start
                    t1_core = core_stop - ext_start

                    LWE_flag_core = LWE_flag_ext[t0_core:t1_core, :, :]

                    time_core = time_idx[core_start:core_stop]
                    lat_sub = lat_vals[i0:i1]
                    lon_sub = lon_vals[j0:j1]

                    da_flag = xr.DataArray(
                        LWE_flag_core,
                        coords={
                            "time": time_core,
                            "latitude": lat_sub,
                            "longitude": lon_sub,
                        },
                        dims=("time", "latitude", "longitude"),
                        name="LWE_flag",
                    )

                    if not os.path.exists(fn_flag):
                        da_flag.to_netcdf(fn_flag)

                    # Save event catalogues.
                    if not df_deep.empty and not os.path.exists(fn_deep):
                        df_deep.to_parquet(fn_deep, index=False)

                    if not df_shal.empty and not os.path.exists(fn_shal):
                        df_shal.to_parquet(fn_shal, index=False)

                    if not df_hubl.empty and not os.path.exists(fn_hubl):
                        df_hubl.to_parquet(fn_hubl, index=False)

                    del (
                        U10_blk,
                        U100_blk,
                        BLH_blk,
                        SSHF_blk,
                        U10_np,
                        U100_np,
                        BLH_np,
                        SSHF_np,
                    )

                    gc.collect()

            print(
                f"[{span_tag}] window {wi}/{len(windows)}: "
                f"{time_idx[t_start]} → {time_idx[t_stop - 1]} finished."
            )

        # Write a completion marker for restartable processing.
        Path(done_flag).write_text("ok", encoding="utf-8")
        print(f"[{span_tag}] DONE")

    print("All spans finished.")


if __name__ == "__main__":
    main()


[1979_1979] processing...
[1979_1979] window 1/1: 1979-01-01 00:00:00 → 1979-12-31 23:00:00 finished.
[1979_1979] DONE
All spans finished.


# Part 2: Inspect and validate the generated LWE event catalogue and LWE flag dataset.

In [ ]:
# 读取数据
def acquire_LWE_data(event_dir, name_of_file):
    # name_of_file: *.parquet, deep_*.parquet
    files = glob.glob(os.path.join(event_dir, name_of_file))
    df_list = [pd.read_parquet(f) for f in files]
    df = pd.concat(df_list, ignore_index=True)
    return df


folder_dir = r"D:\test\1979_1979"

#df_hubonlylow = acquire_LWE_data(folder_dir, name_of_file="hubonlylow_*.parquet")
df_deep = acquire_LWE_data(folder_dir, name_of_file="deep_*.parquet")
df_shallow = acquire_LWE_data(folder_dir, name_of_file="shallow_*.parquet")
df_deep

,lat,lon,start,end,duration_h,kind,delta_u_median,blh_p50,sshf_p50
0,75.0,-23.50,1979-06-09 19:00:00,1979-06-12 09:00:00,63,deep,0.355797,133.753708,2.525556
1,75.0,-23.25,1979-06-09 15:00:00,1979-06-12 10:00:00,68,deep,0.383764,137.656616,3.339028
2,75.0,-23.00,1979-04-16 15:00:00,1979-04-18 14:00:00,48,deep,-0.148826,20.872379,-2.833333
3,75.0,-23.00,1979-06-08 18:00:00,1979-06-13 09:00:00,112,deep,0.379632,106.826813,1.584306
4,75.0,-22.75,1979-04-15 06:00:00,1979-04-18 18:00:00,85,deep,0.080279,20.881557,-2.695000
...,...,...,...,...,...,...,...,...,...
98880,30.0,35.50,1979-12-10 19:00:00,1979-12-13 01:00:00,55,deep,0.292712,27.089846,-3.350833
98881,30.0,35.50,1979-12-29 17:00:00,1979-12-31 16:00:00,48,deep,0.773611,39.316780,-3.261528
98882,30.0,38.25,1979-12-20 20:00:00,1979-12-23 05:00:00,58,deep,0.618517,26.831163,-2.800694
98883,30.0,38.50,1979-12-20 21:00:00,1979-12-22 21:00:00,49,deep,0.303189,30.954849,-2.371944


In [12]:
# 读取数据
def acquire_flag_data(event_dir, name_of_file):


    files = sorted(glob.glob(os.path.join(event_dir, name_of_file)))
    if not files:
        raise FileNotFoundError(f"No files matched: {os.path.join(event_dir, name_of_file)}")

    ds = xr.open_mfdataset(
        files,
        combine="by_coords",
        parallel=False,  
        # chunks="auto",  
    )
    return ds

df_flag = acquire_flag_data(folder_dir, name_of_file="LWEflag_*.nc")
df_flag

<xarray.Dataset> Size: 414MB
Dimensions:    (time: 8760, latitude: 181, longitude: 261)
Coordinates:
  * time       (time) datetime64[ns] 70kB 1979-01-01 ... 1979-12-31T23:00:00
  * latitude   (latitude) float64 1kB 75.0 74.75 74.5 74.25 ... 30.5 30.25 30.0
  * longitude  (longitude) float64 2kB -25.0 -24.75 -24.5 ... 39.5 39.75 40.0
Data variables:
    LWE_flag   (time, latitude, longitude) int8 414MB dask.array<chunksize=(8760, 64, 96), meta=np.ndarray>